# 胚胎发育中的细胞形状分析

## 3DCSQ: 三维细胞形状量化分析系统

本Jupyter Notebook用于分析发育胚胎中的细胞形状，基于球面谐波(SPHARM)和主成分分析(PCA)方法。

### 目录
1. [环境配置与数据加载](#1-环境配置与数据加载)
2. [细胞表面提取](#2-细胞表面提取)
3. [球面谐波变换](#3-球面谐波变换)
4. [PCA降维分析](#4-PCA降维分析)
5. [细胞形状聚类](#5-细胞形状聚类)
6. [谱系树可视化](#6-谱系树可视化)
7. [结果导出](#7-结果导出)

## 1. 环境配置与数据加载

首先导入必要的库并配置环境。

In [1]:
# 导入必要的库
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import os
import nibabel as nib
from scipy import ndimage
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import pyshtools as pysh

# 导入项目自定义模块
import sys
sys.path.append('.')

import utils.cell_func as cell_f
import utils.general_func as general_f
import transformation.SH_represention as sh_represent
from analysis.SH_analyses import analysis_SHcPCA_One_embryo, analysis_SHc_Kmeans_One_embryo

# 设置matplotlib显示中文
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

print("✓ 库导入成功")

✓ 库导入成功


### 配置数据路径

In [2]:
# 配置胚胎数据路径
EMBRYO_DATA_PATH = r'D:\cell_shape_quantification\DATA\SegmentCellUnified04-20\Sample04LabelUnified'
OUTPUT_PATH = r'./results'

# 创建输出目录
os.makedirs(OUTPUT_PATH, exist_ok=True)

# 列出所有时间点的胚胎数据
if os.path.exists(EMBRYO_DATA_PATH):
    embryo_files = [f for f in os.listdir(EMBRYO_DATA_PATH) if f.endswith('.nii.gz')]
    print(f"找到 {len(embryo_files)} 个时间点的数据:")
    for f in sorted(embryo_files)[:10]:  # 显示前10个
        print(f"  - {f}")
    if len(embryo_files) > 10:
        print(f"  ... 还有 {len(embryo_files) - 10} 个文件")
else:
    print(f"⚠️  数据路径不存在：{EMBRYO_DATA_PATH}")
    print("请修改EMBRYO_DATA_PATH为您的实际数据路径")

⚠️  数据路径不存在：D:\cell_shape_quantification\DATA\SegmentCellUnified04-20\Sample04LabelUnified
请修改EMBRYO_DATA_PATH为您的实际数据路径


## 2. 细胞表面提取

从3D分割图像中提取细胞表面点。

In [3]:
# 选择一个时间点进行分析
selected_timepoint = embryo_files[0] if embryo_files else None

if selected_timepoint:
    # 加载胚胎数据
    embryo_path = os.path.join(EMBRYO_DATA_PATH, selected_timepoint)
    img = nib.load(embryo_path)
    img_data = img.get_fdata().astype(np.int16)
    
    print(f"数据形状：{img_data.shape}")
    print(f"数据值范围：{img_data.min()} - {img_data.max()}")
    
    # 获取所有细胞标签
    cell_labels = np.unique(img_data)
    cell_labels = cell_labels[cell_labels != 0]  # 去除背景
    print(f"检测到 {len(cell_labels)} 个细胞")
    
    # 加载细胞名称映射
    try:
        name_dict_path = r'D:\cell_shape_quantification\DATA\name_dictionary_no_name.csv'
        label_name_dict, name_label_dict = cell_f.get_cell_name_affine_table(name_dict_path)
        print("✓ 细胞名称字典加载成功")
    except Exception as e:
        print(f"⚠️  无法加载细胞名称字典：{e}")
        label_name_dict = {i: f'Cell_{i}' for i in cell_labels}

NameError: name 'embryo_files' is not defined

### 提取单个细胞表面并可视化

In [ ]:
# 选择一个细胞进行分析（例如第一个细胞）
if selected_timepoint and len(cell_labels) > 0:
    test_cell_label = cell_labels[0]
    cell_name = label_name_dict.get(test_cell_label, f'Cell_{test_cell_label}')
    
    print(f"分析细胞：{cell_name} (标签：{test_cell_label})")
    
    # 提取细胞表面
    surface_points, center_point = cell_f.nii_get_cell_surface(img_data, test_cell_label)
    
    print(f"表面点数：{len(surface_points)}")
    print(f"细胞中心：{center_point}")
    
    # 将表面点转换到以细胞为中心的坐标系
    surface_points_local = surface_points - center_point
    
    # 3D可视化
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    # 采样显示（避免点太多）
    sample_indices = np.random.choice(len(surface_points_local), min(1000, len(surface_points_local)), replace=False)
    ax.scatter(surface_points_local[sample_indices, 0], 
               surface_points_local[sample_indices, 1], 
               surface_points_local[sample_indices, 2], 
               s=1, alpha=0.5)
    
    ax.set_xlabel('X (μm)')
    ax.set_ylabel('Y (μm)')
    ax.set_zlabel('Z (μm)')
    ax.set_title(f'细胞 {cell_name} 表面点云')
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_PATH, f'{cell_name}_surface.png'), dpi=150)
    plt.show()
    
    print(f"✓ 表面可视化已保存到：{os.path.join(OUTPUT_PATH, f'{cell_name}_surface.png')}")

### 计算细胞体积和表面积

In [ ]:
if selected_timepoint:
    # 计算所有细胞的体积和表面积
    from collections import Counter
    
    volume_dict = {}
    surface_dict = {}
    
    for cell_label in cell_labels:
        # 体积：体素计数
        volume = np.sum(img_data == cell_label)
        volume_dict[cell_label] = volume
        
        # 表面积：使用形态学操作
        cell_mask = (img_data == cell_label)
        dilated = ndimage.binary_dilation(cell_mask)
        surface = np.sum(np.logical_xor(dilated, cell_mask))
        surface_dict[cell_label] = surface
    
    # 创建统计DataFrame
    stats_df = pd.DataFrame({
        'Cell_Label': cell_labels,
        'Cell_Name': [label_name_dict.get(l, f'Cell_{l}') for l in cell_labels],
        'Volume': [volume_dict[l] for l in cell_labels],
        'Surface': [surface_dict[l] for l in cell_labels]
    })
    
    # 计算归一化系数
    stats_df['Norm_Coeff'] = (stats_df['Volume'] / 10000) ** (1/3)
    
    # 显示前10个细胞
    print("细胞体积和表面积统计（前10个）:")
    display(stats_df.head(10))
    
    # 保存统计结果
    stats_df.to_csv(os.path.join(OUTPUT_PATH, 'cell_volume_surface_stats.csv'), index=False)
    print(f"✓ 统计结果已保存到：{os.path.join(OUTPUT_PATH, 'cell_volume_surface_stats.csv')}")

## 3. 球面谐波变换

使用球面谐波(SPHARM)对细胞表面进行参数化和特征提取。

In [ ]:
# 球面采样参数
SAMPLE_N = 30  # 采样点数（纬度方向）
LMAX = 14      # 球面谐波最大阶数

print(f"采样参数：N={SAMPLE_N}, L_max={LMAX}")
print(f"系数数量：{(LMAX + 1) ** 2}")

### 对单个细胞进行SPHARM变换

In [ ]:
if selected_timepoint and len(cell_labels) > 0:
    # 选择第一个细胞进行演示
    demo_cell_label = cell_labels[0]
    demo_cell_name = label_name_dict.get(demo_cell_label, f'Cell_{demo_cell_label}')
    
    # 提取细胞表面
    surface_points, center = cell_f.nii_get_cell_surface(img_data, demo_cell_label)
    points_membrane_local = surface_points - center
    
    # 球面采样
    print(f"正在对细胞 {demo_cell_name} 进行球面采样...")
    griddata, spherical_matrix = sh_represent.do_sampling_with_interval(
        SAMPLE_N, points_membrane_local, average_num=3
    )
    print(f"采样网格形状：{griddata.shape}")
    
    # 计算球面谐波系数
    print("正在计算球面谐波系数...")
    sh_coefficient = pysh.shtools.SHExpandDH(griddata, sampling=2, lmax_calc=LMAX)
    
    # 展平系数
    from utils.sh_cooperation import flatten_clim
    cilm_flat = flatten_clim(sh_coefficient)
    
    print(f"✓ SPHARM系数计算完成，系数维度：{len(cilm_flat)}")
    print(f"前10个系数：{cilm_flat[:10]}")
    
    # 可视化球面谐波系数
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # 系数幅度谱
    ax1.plot(np.abs(cilm_flat))
    ax1.set_xlabel('系数索引')
    ax1.set_ylabel('幅度')
    ax1.set_title(f'{demo_cell_name} 的SPHARM系数幅度谱')
    ax1.grid(True, alpha=0.3)
    
    # 系数相位谱
    ax2.plot(np.angle(cilm_flat))
    ax2.set_xlabel('系数索引')
    ax2.set_ylabel('相位 (rad)')
    ax2.set_title(f'{demo_cell_name} 的SPHARM系数相位谱')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_PATH, f'{demo_cell_name}_spharm_coeffs.png'), dpi=150)
    plt.show()

### 计算整个胚胎所有细胞的SPHARM系数

In [ ]:
if selected_timepoint:
    # 为所有细胞计算SPHARM系数
    spharm_coeffs_dict = {}
    
    print(f"正在计算 {len(cell_labels)} 个细胞的SPHARM系数...")
    
    for idx, cell_label in enumerate(cell_labels):
        cell_name = label_name_dict.get(cell_label, f'Cell_{cell_label}')
        
        # 提取表面
        surface_points, center = cell_f.nii_get_cell_surface(img_data, cell_label)
        points_local = surface_points - center
        
        # 采样
        griddata, _ = sh_represent.do_sampling_with_interval(SAMPLE_N, points_local, average_num=3)
        
        # SPHARM变换
        sh_coeff = pysh.shtools.SHExpandDH(griddata, sampling=2, lmax_calc=LMAX)
        cilm_flat = flatten_clim(sh_coeff)
        
        spharm_coeffs_dict[cell_name] = cilm_flat
        
        if (idx + 1) % 10 == 0:
            print(f"  已处理 {idx + 1}/{len(cell_labels)} 个细胞")
    
    # 转换为DataFrame
    coeff_columns = [f'coeff_{i}' for i in range(len(cilm_flat))]
    spharm_df = pd.DataFrame.from_dict(spharm_coeffs_dict, orient='index', columns=coeff_columns)
    
    print(f"\n✓ 所有细胞的SPHARM系数计算完成")
    print(f"DataFrame形状：{spharm_df.shape}")
    
    # 保存结果
    timepoint_name = selected_timepoint.split('.')[0]
    spharm_df.to_csv(os.path.join(OUTPUT_PATH, f'{timepoint_name}_spharm_coeffs.csv'))
    print(f"✓ 结果已保存到：{os.path.join(OUTPUT_PATH, f'{timepoint_name}_spharm_coeffs.csv')}")

### SPHARM形状重建

In [ ]:
if selected_timepoint and len(cell_labels) > 0:
    from utils.sh_cooperation import do_reconstruction_from_SH, collapse_flatten_clim
    
    # 选择一个细胞进行重建演示
    recon_cell_label = cell_labels[0]
    recon_cell_name = label_name_dict.get(recon_cell_label, f'Cell_{recon_cell_label}')
    
    # 获取原始表面点
    original_surface, center = cell_f.nii_get_cell_surface(img_data, recon_cell_label)
    original_local = original_surface - center
    
    # 使用SPHARM系数重建
    sh_instance = pysh.SHCoeffs.from_array(collapse_flatten_clim(list(spharm_coeffs_dict[recon_cell_name])))
    reconstructed_points = do_reconstruction_from_SH(30, sh_instance)
    
    # 可视化对比
    fig = plt.figure(figsize=(15, 5))
    
    # 原始形状
    ax1 = fig.add_subplot(121, projection='3d')
    sample_idx = np.random.choice(len(original_local), min(1000, len(original_local)), replace=False)
    ax1.scatter(original_local[sample_idx, 0], original_local[sample_idx, 1], 
                original_local[sample_idx, 2], s=2, alpha=0.5, label='原始')
    ax1.set_title(f'{recon_cell_name} - 原始表面')
    ax1.set_xlabel('X')
    ax1.set_ylabel('Y')
    ax1.set_zlabel('Z')
    
    # 重建形状
    ax2 = fig.add_subplot(122, projection='3d')
    ax2.scatter(reconstructed_points[:, 0], reconstructed_points[:, 1], 
                reconstructed_points[:, 2], s=2, alpha=0.5, c='red', label='SPHARM重建')
    ax2.set_title(f'{recon_cell_name} - SPHARM重建 (L={LMAX})')
    ax2.set_xlabel('X')
    ax2.set_ylabel('Y')
    ax2.set_zlabel('Z')
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_PATH, f'{recon_cell_name}_reconstruction.png'), dpi=150)
    plt.show()
    
    print(f"✓ 形状重建对比图已保存")

## 4. PCA降维分析

对SPHARM系数进行主成分分析，提取主要形状特征。

In [ ]:
if 'spharm_df' in locals():
    # 数据标准化
    scaler = StandardScaler()
    spharm_normalized = scaler.fit_transform(spharm_df.values)
    
    # PCA分析
    n_components = min(12, spharm_df.shape[1] - 1)
    pca = PCA(n_components=n_components)
    pca_result = pca.fit_transform(spharm_normalized)
    
    print(f"PCA分析完成")
    print(f"主成分数量：{n_components}")
    print(f"累计解释方差比：{np.sum(pca.explained_variance_ratio_):.3f}")
    print(f"\n各主成分解释方差比:")
    for i, var in enumerate(pca.explained_variance_ratio_):
        print(f"  PC{i+1}: {var:.3f} ({var*100:.1f}%)")
    
    # 创建PCA结果DataFrame
    pca_df = pd.DataFrame(
        pca_result,
        index=spharm_df.index,
        columns=[f'PC{i+1}' for i in range(n_components)]
    )
    
    # 保存PCA结果
    pca_df.to_csv(os.path.join(OUTPUT_PATH, 'pca_components.csv'))
    print(f"\n✓ PCA结果已保存到：{os.path.join(OUTPUT_PATH, 'pca_components.csv')}")

### PCA可视化

In [ ]:
if 'pca_df' in locals():
    # 2D散点图 (PC1 vs PC2)
    plt.figure(figsize=(12, 8))
    plt.scatter(pca_df['PC1'], pca_df['PC2'], alpha=0.6, s=50)
    
    # 添加细胞名称标签
    for idx, cell_name in enumerate(pca_df.index):
        if len(cell_name) <= 4:  # 只标注短名称的细胞
            plt.annotate(cell_name, (pca_df['PC1'][idx], pca_df['PC2'][idx]), 
                        fontsize=8, alpha=0.7)
    
    plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontsize=12)
    plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', fontsize=12)
    plt.title('细胞形状PCA分析 (PC1 vs PC2)', fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_PATH, 'pca_2d_scatter.png'), dpi=150)
    plt.show()
    
    # 3D散点图 (PC1 vs PC2 vs PC3)
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    scatter = ax.scatter(pca_df['PC1'], pca_df['PC2'], pca_df['PC3'], 
                        alpha=0.6, s=50, c=pca_df['PC3'], cmap='viridis')
    
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontsize=12)
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', fontsize=12)
    ax.set_zlabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.1f}%)', fontsize=12)
    ax.set_title('细胞形状PCA分析 (3D)', fontsize=14)
    
    plt.colorbar(scatter, label='PC3值')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_PATH, 'pca_3d_scatter.png'), dpi=150)
    plt.show()
    
    # 解释方差图
    plt.figure(figsize=(10, 6))
    plt.bar(range(1, n_components + 1), pca.explained_variance_ratio_, alpha=0.7)
    plt.plot(range(1, n_components + 1), np.cumsum(pca.explained_variance_ratio_), 
            'ro-', linewidth=2, label='累计解释方差')
    plt.xlabel('主成分', fontsize=12)
    plt.ylabel('解释方差比', fontsize=12)
    plt.title('PCA解释方差分析', fontsize=14)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_PATH, 'pca_explained_variance.png'), dpi=150)
    plt.show()

## 5. 细胞形状聚类

基于PCA特征对细胞进行K-means聚类。

In [ ]:
if 'pca_df' in locals():
    # K-means聚类
    n_clusters = 6  # 可以根据需要调整
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(pca_df.values)
    
    # 添加聚类结果到DataFrame
    pca_df['Cluster'] = cluster_labels
    
    print(f"K-means聚类完成 (K={n_clusters})")
    print(f"\n各聚类的细胞数量:")
    for i in range(n_clusters):
        count = np.sum(cluster_labels == i)
        print(f"  聚类 {i}: {count} 个细胞 ({count/len(cluster_labels)*100:.1f}%)")
    
    # 保存聚类结果
    pca_df.to_csv(os.path.join(OUTPUT_PATH, 'cell_clustering_results.csv'))
    print(f"\n✓ 聚类结果已保存到：{os.path.join(OUTPUT_PATH, 'cell_clustering_results.csv')}")

### 聚类可视化

In [ ]:
if 'pca_df' in locals():
    # 可视化聚类结果
    plt.figure(figsize=(12, 8))
    
    scatter = plt.scatter(pca_df['PC1'], pca_df['PC2'], 
                         c=pca_df['Cluster'], cmap='tab10', 
                         alpha=0.6, s=100, edgecolors='k', linewidth=0.5)
    
    plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontsize=12)
    plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', fontsize=12)
    plt.title(f'细胞形状K-means聚类 (K={n_clusters})', fontsize=14)
    plt.grid(True, alpha=0.3)
    
    # 添加颜色条
    cbar = plt.colorbar(scatter)
    cbar.set_label('聚类标签', fontsize=12)
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_PATH, 'clustering_2d.png'), dpi=150)
    plt.show()
    
    # 3D聚类可视化
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    scatter = ax.scatter(pca_df['PC1'], pca_df['PC2'], pca_df['PC3'], 
                        c=pca_df['Cluster'], cmap='tab10', 
                        alpha=0.6, s=100, edgecolors='k', linewidth=0.5)
    
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontsize=12)
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', fontsize=12)
    ax.set_zlabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.1f}%)', fontsize=12)
    ax.set_title(f'细胞形状K-means聚类 (3D, K={n_clusters})', fontsize=14)
    
    plt.colorbar(scatter, label='聚类标签')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_PATH, 'clustering_3d.png'), dpi=150)
    plt.show()

## 6. 谱系树可视化

绘制细胞谱系树并展示形状特征。

In [ ]:
# 尝试加载谱系树数据
try:
    from lineage_stat.data_structure import get_combined_lineage_tree
    from lineage_stat.lineage_tree import draw_cell_lineage_tree
    
    print("正在加载谱系树数据...")
    cell_tree, begin_frame = get_combined_lineage_tree()
    print(f"✓ 谱系树加载成功")
    print(f"谱系树节点数：{len(cell_tree.all_nodes())}")
    
    # 准备谱系树可视化的值
    if 'pca_df' in locals():
        # 使用PC1值作为谱系树的可视化值
        values_dict = {}
        timepoint_str = selected_timepoint.split('_')[1].split('.')[0] if selected_timepoint else '000'
        
        for cell_name in pca_df.index:
            key = f"{timepoint_str}::{cell_name}"
            values_dict[key] = pca_df.loc[cell_name, 'PC1']
        
        print(f"\n准备绘制谱系树...")
        print(f"可用细胞值：{len(values_dict)} 个")
        
        # 绘制谱系树
        try:
            draw_cell_lineage_tree(
                cell_tree=cell_tree,
                values_dict=values_dict,
                plot_title=f'Embryo_CellShape_PC1_{timepoint_str}',
                color_map='seismic',
                is_frame=True,
                time_resolution=1,
                showing=False,
                path_saving=OUTPUT_PATH
            )
            print(f"✓ 谱系树已保存到：{OUTPUT_PATH}")
        except Exception as e:
            print(f"⚠️  谱系树绘制失败：{e}")
            print("这可能是因为谱系树数据不完整，不影响其他分析结果")
    
except ImportError as e:
    print(f"⚠️  无法导入谱系树模块：{e}")
    print("跳过谱系树可视化步骤")
except Exception as e:
    print(f"⚠️  谱系树分析出错：{e}")
    print("继续其他分析...")

## 7. 结果导出

导出所有分析结果用于后续分析。

In [ ]:
# 汇总所有结果
print("=" * 60)
print("分析结果汇总")
print("=" * 60)

# 1. 细胞统计信息
if 'stats_df' in locals():
    print(f"\n✓ 细胞统计信息:")
    print(f"  - 细胞数量：{len(stats_df)}")
    print(f"  - 平均体积：{stats_df['Volume'].mean():.1f} 体素")
    print(f"  - 平均表面积：{stats_df['Surface'].mean():.1f} 体素")
    stats_df.to_csv(os.path.join(OUTPUT_PATH, 'cell_statistics.csv'), index=False)

# 2. SPHARM系数
if 'spharm_df' in locals():
    print(f"\n✓ SPHARM系数:")
    print(f"  - 系数维度：{spharm_df.shape[1]}")
    print(f"  - 已保存：{timepoint_name}_spharm_coeffs.csv")

# 3. PCA结果
if 'pca_df' in locals():
    print(f"\n✓ PCA分析:")
    print(f"  - 主成分数：{n_components}")
    print(f"  - 累计解释方差：{np.sum(pca.explained_variance_ratio_)*100:.1f}%")
    print(f"  - 已保存：pca_components.csv")

# 4. 聚类结果
if 'pca_df' in locals() and 'Cluster' in pca_df.columns:
    print(f"\n✓ 聚类分析:")
    print(f"  - 聚类数：{n_clusters}")
    for i in range(n_clusters):
        count = np.sum(pca_df['Cluster'] == i)
        cells_in_cluster = pca_df[pca_df['Cluster'] == i].index.tolist()
        print(f"  - 聚类{i}: {count}个细胞 - {', '.join(cells_in_cluster[:5])}{'...' if len(cells_in_cluster) > 5 else ''}")
    print(f"  - 已保存：cell_clustering_results.csv")

print("\n" + "=" * 60)
print(f"所有结果已保存到：{OUTPUT_PATH}")
print("=" * 60)

### 生成分析报告

In [ ]:
# 生成Markdown格式的分析报告
report_content = f"""# 胚胎细胞形状分析报告

## 基本信息
- **分析时间**: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
- **胚胎样本**: {EMBRYO_DATA_PATH.split('\\')[-1]}
- **时间点**: {selected_timepoint if selected_timepoint else 'N/A'}

## 分析参数
- **SPHARM采样数**: {SAMPLE_N}
- **SPHARM最大阶数**: {LMAX}
- **SPHARM系数维度**: {(LMAX + 1) ** 2}
- **PCA主成分数**: {n_components}
- **聚类数**: {n_clusters}

## 主要结果

### 细胞统计
- 总细胞数：{len(stats_df) if 'stats_df' in locals() else 'N/A'}
- 平均体积：{stats_df['Volume'].mean():.1f if 'stats_df' in locals() else 'N/A'} 体素
- 平均表面积：{stats_df['Surface'].mean():.1f if 'stats_df' in locals() else 'N/A'} 体素

### PCA分析
- 累计解释方差：{np.sum(pca.explained_variance_ratio_)*100:.1f if 'pca' in locals() else 'N/A'}%
- PC1解释方差：{pca.explained_variance_ratio_[0]*100:.1f if 'pca' in locals() else 'N/A'}%
- PC2解释方差：{pca.explained_variance_ratio_[1]*100:.1f if 'pca' in locals() else 'N/A'}%

### 聚类分析
"""

if 'pca_df' in locals() and 'Cluster' in pca_df.columns:
    for i in range(n_clusters):
        count = np.sum(pca_df['Cluster'] == i)
        cells = pca_df[pca_df['Cluster'] == i].index.tolist()[:5]
        report_content += f"- 聚类{i}: {count}个细胞 ({', '.join(cells)}...)\n"

report_content += f"""
## 输出文件
- cell_statistics.csv - 细胞体积和表面积统计
- {timepoint_name}_spharm_coeffs.csv - SPHARM系数
- pca_components.csv - PCA主成分
- cell_clustering_results.csv - 聚类结果
- *.png - 可视化图表

## 结论
本分析使用3DCSQ方法对胚胎发育过程中的细胞形状进行了量化分析。
通过SPHARM变换、PCA降维和K-means聚类，成功识别了不同形态特征的细胞群体。
"""

# 保存报告
report_path = os.path.join(OUTPUT_PATH, 'analysis_report.md')
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report_content)

print(f"✓ 分析报告已生成：{report_path}")
print("\n分析完成！")

## 下一步分析建议

1. **时间序列分析**: 分析多个时间点的胚胎数据，追踪细胞形状随时间的变化
2. **细胞谱系追踪**: 结合细胞分裂谱系，研究形状遗传性
3. **差异表达分析**: 识别不同聚类间显著差异的SPHARM系数
4. **功能富集分析**: 研究不同形状类别与细胞命运的关系
5. **机器学习分类**: 使用形状特征预测细胞类型或命运

## 参考文献

1. Li, Z. et al. "3DCSQ: An effective method for quantification, visualization and analysis of 3D cell shape during early embryogenesis"
2. Spherical Harmonics: https://en.wikipedia.org/wiki/Spherical_harmonics
3. PCA: https://en.wikipedia.org/wiki/Principal_component_analysis